In [1]:
# import inputs.vents_grib as vg
import inputs.vents as v
import inputs.polaires as p

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
path = "data/grib_vent/20260422_070337_GFS_P25_00.grb2"
df = v.table(path)
df

,latitude,longitude,valid_time,force,direction,temps,x_data,y_data
0,31.5,0.25,2026-04-22,5.123264,297.945862,0.0,10.5,1890.0
1,31.5,341.25,2026-04-22,4.968140,306.357574,0.0,14332.5,1890.0
2,31.5,341.50,2026-04-22,5.271007,313.458771,0.0,14343.0,1890.0
3,31.5,341.75,2026-04-22,5.255155,322.164795,0.0,14353.5,1890.0
4,31.5,342.00,2026-04-22,4.729717,325.496246,0.0,14364.0,1890.0
...,...,...,...,...,...,...,...,...
405400,47.5,359.00,2026-05-02,6.923615,6.430145,240.0,15078.0,2850.0
405401,47.5,359.25,2026-05-02,6.851637,5.354004,240.0,15088.5,2850.0
405402,47.5,359.50,2026-05-02,7.580052,14.842224,240.0,15099.0,2850.0
405403,47.5,359.75,2026-05-02,8.119076,17.685638,240.0,15109.5,2850.0


In [3]:
VENT_NM = v.vent_grib_nm(path)
VENT_DEG = v.vent_grib_deg(path)

In [4]:
path = "data/polaires/vector.csv"
POLAIRE = p.polaire(path)

In [5]:
%matplotlib qt

In [6]:
dt = 3
n = 100

14.8 * dt * np.pi / n

1.3948671381938684

In [7]:
import cProfile
import core.isochrone as iso

In [8]:
# # p_dep = [46, 2]
# # p_arr = [44, 9]
# # p_dep = [44.53, 360-6.06, 0, 0]
# # p_arr = [48.2, 360-5.6]
# p_dep = [44.52, 360-6.06]
# p_arr = [38.30, 360-12.21]
# p_dep = [p_dep[1] * 60 * 0.7, p_dep[0] * 60, 0, 0]
# p_arr = [p_arr[1] * 60 * 0.7, p_arr[0] * 60]
# t = 0
# def VENT_NM(p, t):
#     return VENT(p, t, 'nm')

# L = iso.n_iso(N = 20, p_dep = p_dep, p_arr = p_arr, t = t, dt = 1, n = 100, V = VENT_NM, P = POLAIRE, r = 0.8, ang = np.pi/2, dang = np.radians(0.9), delta = 1)

In [9]:
# p_dep = [46, 2]
# p_arr = [44, 9]
# p_dep = [44, 360-2]
# p_arr = [48.2, 360-5.6]
p_dep = [36, 360-10]
p_arr = [36, 360-2]
t = 100

routage = iso.routage(p_dep, p_arr, t, dt = 0.5, n = 100, V = VENT_NM, P = POLAIRE, e_arr = 5, r = 1, ang = np.pi/2, dang = np.radians(0.1), delta = 2)

# cProfile.run("routage = iso.routage(p_dep, p_arr, t, dt = 0.25, n = 100, V = VENT, P = POLAIRE, e_arr = 5, r = 0.5, ang = np.pi/2, dang = np.radians(0.4), delta = 0.1)",
#             sort="tottime")


nombre isochrones : 0, temps : 100.00 heures, nombre de points : 1
nombre isochrones : 1, temps : 100.50 heures, nombre de points : 100
nombre isochrones : 2, temps : 101.00 heures, nombre de points : 82
nombre isochrones : 3, temps : 101.50 heures, nombre de points : 110
nombre isochrones : 4, temps : 102.00 heures, nombre de points : 137
nombre isochrones : 5, temps : 102.50 heures, nombre de points : 165
nombre isochrones : 6, temps : 103.00 heures, nombre de points : 196
nombre isochrones : 7, temps : 103.50 heures, nombre de points : 223
nombre isochrones : 8, temps : 104.00 heures, nombre de points : 254
nombre isochrones : 9, temps : 104.50 heures, nombre de points : 280
nombre isochrones : 10, temps : 105.00 heures, nombre de points : 299
nombre isochrones : 11, temps : 105.50 heures, nombre de points : 318
nombre isochrones : 12, temps : 106.00 heures, nombre de points : 336
nombre isochrones : 13, temps : 106.50 heures, nombre de points : 349
nombre isochrones : 14, temps : 1

In [17]:
latitude, longitude, time_list, L = routage

t_arr = time_list[-1]
print((t_arr - t) // 24, "jours", (t_arr - t) % 24, "heures")

6.0 jours 21.5 heures


In [11]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(6,6))
ax = plt.axes(projection=ccrs.Mercator())

ax.add_feature(cfeature.BORDERS, linewidth=1)
ax.add_feature(cfeature.COASTLINE, linewidth=1)

gl = ax.gridlines(draw_labels=True,
                  crs=ccrs.PlateCarree(),
                  linestyle='--',
                  alpha=0.6)

gl.top_labels = False
gl.right_labels = False

# -------------------------
# GRILLE
# -------------------------
V = VENT_DEG
t = 6

intx = [350, 360]
inty = [42, 49]
n = 60

lon = np.linspace(intx[0], intx[1], n+1)
lat = np.linspace(inty[0], inty[1], n+1)

X, Y = np.meshgrid(lon, lat)

# -------------------------
# CALCUL DU VENT
# -------------------------
U = np.zeros_like(X)
Vv = np.zeros_like(Y)
C = np.zeros_like(X)

for i in range(n+1):
    for j in range(n+1):
        direction, speed = V([X[i,j], Y[i,j]], t)
        
        U[i,j] = - speed * np.sin(np.radians(direction))
        Vv[i,j] = - speed * np.cos(np.radians(direction))
        C[i,j] = speed

# -------------------------
# QUIVER
# -------------------------
q = ax.quiver(X, Y, U, Vv, C,
              transform=ccrs.PlateCarree(),
              cmap='Greys')

ax.set_extent([intx[0], intx[1], inty[0], inty[1]],
              crs=ccrs.PlateCarree())

# route 
# ax.plot(longitude, latitude, transform=ccrs.PlateCarree(), c = 'red')
# ax.scatter(longitude, latitude, transform=ccrs.PlateCarree(), c = 'red')

plt.show()

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime, timedelta

# -------------------------
# Paramètres de la grille
# -------------------------
V = VENT_DEG  # votre fonction de vent

# intx = [min(pd.unique(df['longitude'])), max(pd.unique(df['longitude']))]
inty = [min(pd.unique(df['latitude'])), max(pd.unique(df['latitude']))]
intx = [341.25, max(pd.unique(df['longitude']))]
intt = [0, max(pd.unique(df['temps']))]
n = 100

start_date = df['valid_time'][0]

lon = np.linspace(intx[0], intx[1], n+1)
lat = np.linspace(inty[0], inty[1], n+1)
X, Y = np.meshgrid(lon, lat)

# -------------------------
# Initialisation figure
# -------------------------
fig = plt.figure(figsize=(6,6))
ax = plt.axes(projection=ccrs.Mercator())
# ax.add_feature(cfeature.BORDERS, linewidth=1)
ax.add_feature(cfeature.COASTLINE, linewidth=1)
# ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m'), linewidth=1)

gl = ax.gridlines(draw_labels=True, crs=ccrs.PlateCarree(),
                  linestyle='--', alpha=0.6)
gl.top_labels = False
gl.right_labels = False

ax.set_extent([intx[0], intx[1], inty[0], inty[1]],
              crs=ccrs.PlateCarree())

# -------------------------
# Fonction pour calculer vent
# -------------------------
def compute_wind(t):
    U = np.zeros_like(X)
    Vv = np.zeros_like(Y)
    C = np.zeros_like(X)
    for i in range(n+1):
        for j in range(n+1):
            direction, speed = VENT_DEG([X[i,j], Y[i,j]], t)
            U[i,j] = - speed * np.sin(np.radians(direction))
            Vv[i,j] = - speed * np.cos(np.radians(direction))
            if speed > 20:
                C[i,j] = 20
            else:
                C[i,j] = speed
    return U, Vv, C

# -------------------------
# Vent initial t=0
# -------------------------
initial_time = intt[0]
U, Vv, C = compute_wind(initial_time)
q = ax.quiver(X, Y, U, Vv, C, transform=ccrs.PlateCarree(), cmap='Greys')


# -------------------------
# Route
# -------------------------
for i in range(len(latitude)):
    l = L[L[:, 2] == i]
    ax.plot(l[:, 1], l[:, 0], transform=ccrs.PlateCarree(), c = 'lightcoral')
    
route_line, = ax.plot(longitude, latitude, transform=ccrs.PlateCarree(), c='red')
boat_scatter = ax.scatter(latitude[0], longitude[0], c='red', transform=ccrs.PlateCarree())


# -------------------------
# Slider
# -------------------------
ax_slider = plt.axes([0.2, 0.02, 0.6, 0.03])
slider = Slider(ax_slider, 'Temps', valmin=intt[0], valmax=intt[1], valinit=intt[0], valstep=1, facecolor='grey')
slider.vline.set_visible(False)

initial_date = start_date + timedelta(hours=initial_time)
slider.valtext.set_text(initial_date.strftime("%d/%m %H:%M"))

def update(val):
    t = slider.val
    U, Vv, C = compute_wind(t)
    q.set_UVC(U, Vv, C)

    # date
    current_date = start_date + timedelta(hours=t)
    slider.valtext.set_text(current_date.strftime("%d/%m %H:%M"))

    # Route
    idx = np.argmin(np.abs(time_list - t))
    boat_scatter.set_offsets([[longitude[idx], latitude[idx]]])


    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
ax = plt.axes(projection=ccrs.Mercator())
ax.add_feature(cfeature.COASTLINE, linewidth=1)
# ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m'), linewidth=1

In [ ]:
import cartopy.io.shapereader as shpreader
from shapely import contains_xy
from shapely.ops import unary_union

# Charge les polygones de terre Natural Earth
land_shp = shpreader.natural_earth(
    resolution='10m',
    category='physical',
    name='land'
)

reader = shpreader.Reader(land_shp)
land_geom = unary_union(list(reader.geometries()))  # fusionne tous les polygones

def points_sur_terre_ou_mer(coords):
    coords[:, 0], coords[:, 1] = coords[:, 1] / 60, coords[:, 0] / (60 * 0.7)
    lat = coords[:, 0]
    lon = coords[:, 1]

    # Test vectorisé
    return ~contains_xy(land_geom, lon, lat)

In [ ]:
import cartopy.io.shapereader as shpreader
from shapely.geometry import Point
from shapely.ops import unary_union

# Charge les polygones de terre Natural Earth
land_shp = shpreader.natural_earth(
    resolution='10m',
    category='physical',
    name='land'
)

reader = shpreader.Reader(land_shp)
land_geom = unary_union(list(reader.geometries()))  # fusionne tous les polygones

def valide_nm(x, y):
    lat, lon = y / 60, x / (60 * 0.7)
    p = Point(lon, lat)
    return not land_geom.contains(p)

def valide_deg(lat, lon):
    lon = lon - 360
    p = Point(lon, lat)
    return not land_geom.contains(p)

In [ ]:
valide_deg(46, 358)

In [ ]:
valide_nm(0 * 60 * 0.7, 46 * 60)

In [ ]:
VENT([46, 358], 6, 'deg')

In [ ]:
import cartopy.io.shapereader as shpreader
from shapely.geometry import Point
from shapely.ops import unary_union

# Charge les polygones de terre Natural Earth
land_shp = shpreader.natural_earth(
    resolution='10m',
    category='physical',
    name='land'
)

reader = shpreader.Reader(land_shp)
land_geom = unary_union(list(reader.geometries()))  # fusionne tous les polygones

def point_sur_terre_ou_mer(lat, lon):
    p = Point(lon, lat)   # attention : Point(x, y) = Point(lon, lat)
    return not land_geom.contains(p)

In [ ]:
point_sur_terre_ou_mer(46, 359)

In [ ]:
import pandas as pd
import numpy as np

def table_terre_mer(lat_start, lat_end, lon_start, lon_end, lat_step, lon_step):
    lats = np.arange(lat_start, lat_end + lat_step, lat_step)
    lons = np.arange(lon_start, lon_end + lon_step, lon_step)
    lat_grid, lon_grid = np.meshgrid(lats, lons, indexing='ij')
    flat_lats = lat_grid.ravel()
    flat_lons = lon_grid.ravel()
    # Utilise numpy pour la grille, mais reste une boucle sur le point_sur_terre_ou_mer
    values = np.array([point_sur_terre_ou_mer(lat, lon) for lat, lon in zip(flat_lats, flat_lons)])
    df = pd.DataFrame({
        "latitude": flat_lats,
        "longitude": flat_lons,
        "terre_mer": values
    })
    return df

# Exemple d'utilisation :
df = table_terre_mer(43.0, 50.0, -8.0, 0.0, 0.05, 0.05)
print(df)

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
scatter = ax.scatter(df["longitude"], df["latitude"], c=df["terre_mer"], cmap="bwr", marker="o", s=10, transform=ccrs.PlateCarree())
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Points Terre/Mer (0 = Terre, 1 = Mer)")
plt.colorbar(scatter, ticks=[0, 1], label="Terre (0) / Mer (1)")
ax.add_feature(cfeature.COASTLINE, linewidth=1)
plt.show()

In [ ]:
import pandas as pd

# On ouvre le fichier CSV data/polaires/vector.csv et on l'affiche
df = pd.read_csv("data/polaires/vector.csv")
df

,TWA/TWS,0,6,8,12,16,20,25,30,40
0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,10,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,20,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,30,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,40,0,2.7,3.5,4.0,5.0,5.2,2.7,2.3,2.3
5,50,0,3.9,4.6,5.5,6.2,6.6,5.0,3.8,3.8
6,60,0,4.7,5.7,6.5,7.2,7.4,7.0,6.2,6.2
7,70,0,5.0,6.1,7.1,7.9,8.1,8.4,8.0,8.0
8,80,0,5.2,6.3,7.8,9.0,9.7,10.1,9.6,9.6
9,90,0,5.2,6.4,8.3,9.9,11.2,11.6,10.6,10.6


In [ ]:
df.iloc[:, 0]

0       0
1      10
2      20
3      30
4      40
5      50
6      60
7      70
8      80
9      90
10    100
11    110
12    120
13    130
14    140
15    150
16    160
17    170
18    180
Name: TWA/TWS, dtype: int64

In [ ]:
df.iloc[0, 1:]

0     0.0
6     0.0
8     0.0
12    0.0
16    0.0
20    0.0
25    0.0
30    0.0
40    0.0
Name: 0, dtype: float64

In [ ]:
df.iloc[:, 1:]

,0,6,8,12,16,20,25,30,40
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,2.7,3.5,4.0,5.0,5.2,2.7,2.3,2.3
5,0,3.9,4.6,5.5,6.2,6.6,5.0,3.8,3.8
6,0,4.7,5.7,6.5,7.2,7.4,7.0,6.2,6.2
7,0,5.0,6.1,7.1,7.9,8.1,8.4,8.0,8.0
8,0,5.2,6.3,7.8,9.0,9.7,10.1,9.6,9.6
9,0,5.2,6.4,8.3,9.9,11.2,11.6,10.6,10.6
